<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/transformers_agents_multiagents/Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#pip install -qU langchain  langchain-cohere langchain_community

In [ ]:
import getpass
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "test-project"
os.environ["USER_AGENT"] = "LangChain/1.0.0"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()

 ········


In [ ]:
os.environ["COHERE_API_KEY"] = getpass.getpass()

 ········


In [ ]:
os.environ["TAVILY_API_KEY"] = getpass.getpass()

 ········


In [ ]:
from langchain_cohere import ChatCohere

C:\Users\Ihechi Festus\Documents\ML\ml_env\Lib\site-packages\pydantic\_internal\_config.py:341: UserWarning: Valid config keys have changed in V2:
* 'allow_population_by_field_name' has been renamed to 'populate_by_name'
* 'smart_union' has been removed
  warnings.warn(message, UserWarning)


In [ ]:
model = ChatCohere(model="command-r-plus")

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
store = {}

In [ ]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(
    model, get_session_history,
)

In [ ]:
config = {
    "configurable": {
        "session_id": "abc2"
    }
}

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
from langchain_core.messages import trim_messages

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# This uses a trimmer, messages are trimmed as the length increases, resulting in the model forgetting earlier messages.
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability."
        ),
        MessagesPlaceholder(variable_name="messages")
    ]
)

trimmer = trim_messages(
    max_tokens=65,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

chain = (
    RunnablePassthrough.assign(
        messages=itemgetter("messages") | trimmer
    ) | prompt | model
)

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)

In [ ]:
c = prompt.invoke({"messages": ["Hello chatgpt", "How are you"]})

In [ ]:
c.to_messages()

[SystemMessage(content='You are a helpful assistant. Answer all questions to the best of your ability.'),
 HumanMessage(content='Hello chatgpt'),
 HumanMessage(content='How are you')]

In [ ]:
for r in with_message_history.stream(
    {
        "messages": ["Hi! I'm Bob. Tell me a joke"],
    },
    config=config,
):
    print(r.content, end="|")


Why| did| the| tomato| turn| red|?|

Because| it| saw| the| salad| dressing|!||

In [ ]:
for r in with_message_history.stream(
    {
        "messages": ["What is my name"],
    },
    config=config,
):
    print(r.content, end="|")

Your| name| is| Bob|!||

In [ ]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

In [ ]:
import bs4
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# 1. Load, chunk and index the contents of the blog to create a retriever.
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vectorstore = InMemoryVectorStore.from_documents(
    documents=splits, embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

In [ ]:
# Build context Prompt
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    model, retriever, contextualize_q_prompt
)

NameError: name 'retriever' is not defined

In [ ]:
# Build the agent
memory = MemorySaver()
search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_react_agent(model, tools, checkpointer=memory)

NameError: name 'MemorySaver' is not defined

In [ ]:
# Use the agent
config = {"configurable": {"thread_id": "abc123"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="hi im bob! and i live in sf")]}, config
):
    print(chunk)
    print("----")


In [ ]:
from langchain_core.runnables import ConfigurableFieldSpec


def get_session_history(user_id: str, conversation_id: str):
    return SQLChatMessageHistory(f"{user_id}--{conversation_id}", "sqlite:///memory.db")


with_message_history = RunnableWithMessageHistory(
    runnable,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="user_id",
            annotation=str,
            name="User ID",
            description="Unique identifier for the user.",
            default="",
            is_shared=True,
        ),
        ConfigurableFieldSpec(
            id="conversation_id",
            annotation=str,
            name="Conversation ID",
            description="Unique identifier for the conversation.",
            default="",
            is_shared=True,
        ),
    ],
)

with_message_history.invoke(
    {"language": "italian", "input": "hi im bob!"},
    config={"configurable": {"user_id": "123", "conversation_id": "1"}},
)

In [ ]:
chunks = []
async for chunk in with_message_history.astream("what color is the sky?"):
    chunks.append(chunk)
    print(chunk.content, end="|", flush=True)

In [ ]:
chunks[0] + chunks[1] + chunks[2] + chunks[3] + chunks[4]

In [ ]:
import time

In [ ]:
s = "The quick brown fox jumps over the lazy dog"

chunks = []
for w in s.split(" "):
    chunks.append(w)
    print(" ".join(chunks), end="\r")
    time.sleep(2)

In [ ]:
print("Hello again", end="\r")
print("Hellor")

Helloragain


In [ ]:
10.00 - 10.25
18 - 19.05

sunday
yivera 19 -20
- 00.35

monday
17.45 - 9.15

In [ ]:
# Import relevant functionality
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

# Create the agent
memory = MemorySaver()
search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_react_agent(model, tools, checkpointer=memory)

# Use the agent
config = {"configurable": {"thread_id": "abc123"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="hi im bob! and i live in sf")]}, config
):
    print(chunk)
    print("----")


In [ ]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    model, retriever, contextualize_q_prompt
)

In [ ]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(model, qa_prompt)
rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain,
) # A rag chain that combines a retriever and a question answering chains

In [ ]:
from langchain import hub

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.messages

C:\Users\Ihechi Festus\Documents\ML\ml_env\Lib\site-packages\langsmith\client.py:5402: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(json.dumps(prompt_object.manifest))


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a helpful assistant')),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [ ]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

default_system_prompt = (
    "You are a helpful assistant"
)

default_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", default_system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)

summarization_prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="chat_history"),
        (
            "user",
            "Distill the above chat messages into a single summary message. Include as many specific details as you can.",
        ),
    ]
)

In [ ]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
from collections import defaultdict

In [ ]:
store

defaultdict(dict,
            {None: {None: InMemoryChatMessageHistory(messages=[])},
             'a': {'b': InMemoryChatMessageHistory(messages=[])}})

In [ ]:
get_session_history("b", "c")

InMemoryChatMessageHistory(messages=[])

In [ ]:
store

defaultdict(dict,
            {'a': {'b': InMemoryChatMessageHistory(messages=[]),
              'c': InMemoryChatMessageHistory(messages=[])},
             'b': {'c': InMemoryChatMessageHistory(messages=[])}})

In [ ]:
store = defaultdict(dict, {})

In [ ]:
c = store["a"]["b"]

In [ ]:
c.add_message("I have a ph.d in Jolly")

In [ ]:
c

InMemoryChatMessageHistory(messages=[AIMessage(content='Your name is Bob, and you live in Botswana. You have expressed an interest in traveling to America and a like for AI.', additional_kwargs={'documents': None, 'citations': None, 'search_results': None, 'search_queries': None, 'is_search_required': None, 'generation_id': '85e0e32a-4c24-40ec-b727-aa7560c2c939', 'token_count': {'input_tokens': 256.0, 'output_tokens': 26.0}}, response_metadata={'documents': None, 'citations': None, 'search_results': None, 'search_queries': None, 'is_search_required': None, 'generation_id': '85e0e32a-4c24-40ec-b727-aa7560c2c939', 'token_count': {'input_tokens': 256.0, 'output_tokens': 26.0}}, id='run-833f9d6c-f029-4020-a052-33ef1cec132f-0', usage_metadata={'input_tokens': 256, 'output_tokens': 26, 'total_tokens': 282}), "My mother's name is Martha", "My father's name is John", 'This is fun', 'I have a ph.d in Jolly'])

In [ ]:
summarize_messages({"input": "hello"}, {"configurable": {"user_id": "a",
                    "conversation_id": "b"
                                                        }
                                       }
                  )

[AIMessage(content="You are Bob from Botswana, and you have a Ph.D. in Jolly. Your parents' names are Martha and John, and you've expressed a desire to travel to America, showing an interest in AI as well.", additional_kwargs={'documents': None, 'citations': None, 'search_results': None, 'search_queries': None, 'is_search_required': None, 'generation_id': '339f4483-5601-4b41-8ebe-c9e1f02385d9', 'token_count': {'input_tokens': 283.0, 'output_tokens': 45.0}}, response_metadata={'documents': None, 'citations': None, 'search_results': None, 'search_queries': None, 'is_search_required': None, 'generation_id': '339f4483-5601-4b41-8ebe-c9e1f02385d9', 'token_count': {'input_tokens': 283.0, 'output_tokens': 45.0}}, id='run-05eebecf-0a53-4a86-b5a5-1547143ad96d-0', usage_metadata={'input_tokens': 283, 'output_tokens': 45, 'total_tokens': 328})]

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

# Create the agent
search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_react_agent(model, tools)

In [ ]:
from langchain_core.runnables import ConfigurableFieldSpec
store = defaultdict(dict, {})

def get_session_history(user_id: str, conversation_id: str) -> BaseChatMessageHistory:
    if conversation_id not in store[user_id]:
        store[user_id][conversation_id] = ChatMessageHistory()

    return store[user_id][conversation_id]

def summarize_messages(chain_input, config):
    user_id = config.get("configurable", {}).get("user_id")
    conversation_id = config.get("configurable", {}).get("conversation_id")
    checkpoint = 10

    session_history = get_session_history(user_id, conversation_id)
    stored_messages = session_history.messages

    # summarize conversations if len of conversation greater than checkpoint.
    # offset by 1 to account for AImessage which holds previous summaries
    if len(stored_messages) >= checkpoint + 1:
        summarization_chain = summarization_prompt | model

        summary_message = summarization_chain.invoke(
            {
                "chat_history": stored_messages
            }
        )
        session_history.clear()
        session_history.add_message(summary_message)

    return session_history.messages

agent_with_chat_history  = RunnableWithMessageHistory(
    default_prompt | model,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    history_factory_config=[
        ConfigurableFieldSpec(
            id="user_id",
            annotation=str,
            name="User ID",
            description="Unique identifier for the user.",
            default="",
            is_shared=True,
        ),
        ConfigurableFieldSpec(
            id="conversation_id",
            annotation=str,
            name="Conversation ID",
            description="Unique identifier for the conversation.",
            default="",
            is_shared=True,
        ),
    ],
)

chain_with_summarization = (
    RunnablePassthrough.assign(chat_history=lambda chain_input, config: summarize_messages(chain_input, config))
    | RunnablePassthrough.assign(m=see_input)
    | agent_with_chat_history
)

# agent_with_chat_history.invoke(
#     {"language": "italian", "input": "hi im bob!"},
#     config={"configurable": {"user_id": "123", "conversation_id": "1"}},
# )

In [ ]:
def see_input(chain_input):
    print("Chain input: ", chain_input)
    return chain_input

In [ ]:
chain_with_summarization.invoke(
    {"input": "Alright, that is understandable", },
    config={"configurable": {"user_id": "a", "conversation_id": "b"}}
)

Chain input:  {'input': 'Alright, that is understandable', 'chat_history': [AIMessage(content="A user asked about their name, but I clarified that I don't have that information and encouraged them to provide it. They then inquired about an attempted assassination of Donald Trump this year, but I explained that my knowledge is cutoff as of January 2023, and I don't have records of any such event within that timeframe. The discussion ended with a question about Donald Trump's running mate, to which I again stated my knowledge limitations and couldn't provide an answer.", additional_kwargs={'documents': None, 'citations': None, 'search_results': None, 'search_queries': None, 'is_search_required': None, 'generation_id': '321b91c7-9a48-421a-97d3-abb30b09e59e', 'token_count': {'input_tokens': 501.0, 'output_tokens': 97.0}}, response_metadata={'documents': None, 'citations': None, 'search_results': None, 'search_queries': None, 'is_search_required': None, 'generation_id': '321b91c7-9a48-421a-

AIMessage(content='Is there anything else I can help you with?', additional_kwargs={'documents': None, 'citations': None, 'search_results': None, 'search_queries': None, 'is_search_required': None, 'generation_id': '552057ad-f523-4232-aa1b-f53500cd1039', 'token_count': {'input_tokens': 312.0, 'output_tokens': 10.0}}, response_metadata={'documents': None, 'citations': None, 'search_results': None, 'search_queries': None, 'is_search_required': None, 'generation_id': '552057ad-f523-4232-aa1b-f53500cd1039', 'token_count': {'input_tokens': 312.0, 'output_tokens': 10.0}}, id='run-ca94f4db-e69c-443f-aea4-06de88fa1905-0', usage_metadata={'input_tokens': 312, 'output_tokens': 10, 'total_tokens': 322})

In [ ]:
My cutoff date for knowledge is January 2023

In [ ]:
c = store["a"]["b"]

In [ ]:
for message in c.messages:
    if isinstance(message, AIMessage):
        prefix = "AI"
    else:
        prefix = "User"

    print(f"{prefix}: {message.content}\n")

AI: A user asked about their name, but I clarified that I don't have that information and encouraged them to provide it. They then inquired about an attempted assassination of Donald Trump this year, but I explained that my knowledge is cutoff as of January 2023, and I don't have records of any such event within that timeframe. The discussion ended with a question about Donald Trump's running mate, to which I again stated my knowledge limitations and couldn't provide an answer.

User: Alright, that is understandable

AI: Is there anything else I can help you with?



In [ ]:
len(c.messages)

10